In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import os
import matplotlib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
dataset = os.path.join('..', '..', '..', 'datasets', 'hotel_bookings.csv')
df = pd.read_csv(dataset)
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['is_canceled'].value_counts().plot(kind='bar')

In [ ]:
df['is_cenceled'].value_counts()/df['is_canceled'].count()*100

In [ ]:
df.corr()

In [ ]:
df.corr()['is_canceled'].sort_values(ascending=True)

In [ ]:
x, y = df.iloc[:, 1:], df.iloc[:, 0]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=123)
xgb_clf = xgb.XGBClassifier(random_state=123)
xgb_clf.get_params()

In [ ]:
xgb_clf.set_params(n_estimators=10)
xgb_clf.fit(x_train, y_train)
preds = xgb_clf.predict(x_test)

In [ ]:
accuracy = float(np.sum(preds==y_test))/y_test.shape[0]
print('Baseline accuracy:', accuracy)

In [ ]:
matplotlib.rcParams['figure.figsize'] = (10, 8)
xgb.plot_importance(xgb_clf)

In [ ]:
matplotlib.rcParams['figure.figsize'] = (20, 8)
xgb.plot_tree(xgb_clf, num_trees=0)

In [ ]:
bookings_dmatrix = xgb.DMatrix(data=x, label=y)

In [ ]:
params = {'objective': 'binary:logistic', 'max_depth': 3}
xgb_cv = xgb.cv(dtrain=bookings_dmatrix, params=params, nfold=3, num_boost_round=10, seed=123, metrics=['error'])
xgb_cv

In [ ]:
accuracy = 1 - xgb_cv['test-error-mean'].iloc[-1]
print('Baseline cv accuracy:', accuracy)

In [ ]:
xgb_cv = xgb.cv(dtrain=bookings_dmatrix, params=params, nfold=3, num_boost_round=40,early_stopping_rounds=10, seed=123, metrics=['error'])
accuracy = 1 - xgb_cv['test-error-mean'].iloc[-1]
print('accuracy:', accuracy)

In [ ]:
xgb_cv

In [ ]:
xgb_clf = xgb.XGBClassifier(n_estimators=25, random_state=123)

In [ ]:
xgb_clf.set_params(max_depth=10)
xgb_clf.fit(x_train, y_train)
preds = xgb_clf.predict(x_test)
accuracy_score(y_test, preds)

In [ ]:
xgb_clf.set_params(colsample_bytree=0.5)
xgb_clf.fit(x_train, y_train)
preds = xgb_clf.predict(x_test)
accuracy_score(y_test, preds)

In [ ]:
xgb_clf.set_params(subsample=0.5)
xgb_clf.fit(x_train, y_train)
preds = xgb_clf.predict(x_test)
accuracy_score(y_test, preds)

In [ ]:
xgb_clf.set_params(gamma=0.25)
xgb_clf.fit(x_train, y_train)
preds = xgb_clf.predict(x_test)
accuracy_score(y_test, preds)

In [ ]:
xgb_clf.set_params(learning_rate=0.3)
xgb_clf.fit(x_train, y_train)
preds = xgb_clf.predict(x_test)
accuracy_score(y_test, preds)

In [ ]:
xgb_clf.set_params(reg_alpha=0.01)
xgb_clf.fit(x_train, y_train)
preds = xgb_clf.predict(x_test)
accuracy_score(y_test, preds)

In [ ]:
xgb_clf.get_params()

In [ ]:
rs_params_grid = {
    'max_depth': list((range(3, 12))),
    'alpha': [0, 0.001, 0.01, 0.1, 1],
    'subsample': [0.5, 0.75, 1],
    'learning_rate': np.linspace(0.01, 0.5, 10),
    'n_estimators': [10, 25, 40]
}

xgb_clf = xgb.XGBClassifier(random_state=123)
xgb_rs = RandomizedSearchCV(estimator=xgb_clf, param_distributions=rs_params_grid, cv=3, n_iter=5, verbose=2, random_state=123)
xgb_rs.fit(x_train, y_train)
print('Best parameters found:' xgb_rs.best_params_)
print('Best accuracy found:' xgb_rs.best_score_)